# Lecture 14
## Network training

We will look at factors that influence how well the networks train and what might cause difficulties.

### Example for vanishing gradients

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# 1. Generate Synthetic Deep Financial Risk Data
# 1000 samples, 50 macroeconomic/accounting features, binary classification (default/no-default)
torch.manual_seed(42)
X = torch.randn(1000, 50)
y = torch.randint(0, 2, (1000, 1)).float()

# 2. Define a Dynamically Configurable 10-Layer Deep MLP
class DeepMLP(nn.Module):
    def __init__(self, input_dim=50, hidden_dim=64, num_layers=10, activation='sigmoid'):
        super(DeepMLP, self).__init__()
        layers = []
        
        # Select activation function
        act_fn = nn.Sigmoid() if activation == 'sigmoid' else nn.ReLU()
        
        # Input layer
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(act_fn)
        
        # Hidden layers (Layers 2 through 9)
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(act_fn)
            
        # Output layer (Layer 10) -> Outputs single probability logit
        layers.append(nn.Linear(hidden_dim, 1))
        
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

# 3. Diagnostic Hook: Extract Gradient Norms Per Layer
def get_gradient_norms(model):
    """
    Filters out linear layers and extracts the L2 norm of 
    the gradients with respect to weights after backward() is called.
    """
    grad_norms = []
    # Interrogate layers from input (Layer 1) to output (Layer 10)
    for layer in model.network:
        if isinstance(layer, nn.Linear):
            if layer.weight.grad is not None:
                # Calculate L2 norm: sqrt(sum(g^2))
                norm = layer.weight.grad.norm(2).item()
                grad_norms.append(norm)
    return grad_norms

# 4. Run the Diagnostic Experiment
def run_experiment():
    # Instantiate models, loss, and run a single backward pass
    sigmoid_model = DeepMLP(activation='sigmoid')
    relu_model = DeepMLP(activation='relu')
    criterion = nn.BCEWithLogitsLoss()
    
    # --- Step A: Sigmoid Forward & Backward Pass ---
    out_sig = sigmoid_model(X)
    loss_sig = criterion(out_sig, y)
    sigmoid_model.zero_grad()
    loss_sig.backward()
    sigmoid_grads = get_gradient_norms(sigmoid_model)
    
    # --- Step B: ReLU Forward & Backward Pass ---
    out_relu = relu_model(X)
    loss_relu = criterion(out_relu, y)
    relu_model.zero_grad()
    loss_relu.backward()
    relu_grads = get_gradient_norms(relu_model)
    
    # 5. Plot the Real-Time Signal Decay
    layers = list(range(1, 11))
    
    plt.figure(figsize=(10, 6))
    plt.plot(layers, sigmoid_grads, marker='o', color='crimson', linewidth=2, label='Sigmoid Model (Vanishing Gradient)')
    plt.plot(layers, relu_grads, marker='s', color='teal', linewidth=2, label='ReLU Model (Stable Flow)')
    
    plt.title('Gradient Magnitude Across a 10-Layer MLP', fontsize=14, fontweight='bold')
    plt.xlabel('Layer Number (Input → Output)', fontsize=12)
    plt.ylabel('Gradient L2 Norm (Log Scale)', fontsize=12)
    plt.yscale('log') # Vital for visualizing exponential decay
    plt.xticks(layers)
    plt.grid(True, which="both", linestyle="--", alpha=0.5)
    plt.legend(fontsize=11)
    
    # Annotate details for the slide/screen
    plt.annotate('Signal Peak (Loss Output)', xy=(10, sigmoid_grads[-1]), xytext=(7.5, sigmoid_grads[-1]*1.5),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))
    plt.annotate('Signal Dead (~10⁻⁶)', xy=(1, sigmoid_grads[0]), xytext=(2, sigmoid_grads[0]*0.1),
                 arrowprops=dict(facecolor='crimson', shrink=0.05, width=1, headwidth=6), color='crimson')

    plt.show()

run_experiment()


### Example for financial asset pricing

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# 1. Generate Synthetic Asset Pricing Data (Tail Risk & Negative Returns)
# 1500 trading days, 4 features: [Market Return, Volatility Delta, Macro Momentum, Credit Spread]
torch.manual_seed(101)
np.random.seed(101)

X_raw = np.random.randn(1500, 4)
# Intentionally inject extreme market anomalies / negative shocks
X_raw[::10, 0] *= 4.5  # Extreme market crashes or rallies (outliers)
X_raw[::7, 1] -= 3.0   # Sharp sudden drops in volatility delta (negative inputs)

X = torch.tensor(X_raw, dtype=torch.float32)
# Target: Future Asset Option Premium (highly non-linear interaction with heavy tails)
y = (X[:, 0] * 1.5 + torch.clamp(X[:, 1], min=-1.0) * X[:, 2]**2 + torch.exp(X[:, 3] * 0.5)).view(-1, 1)

# 2. Architectures: Asset Pricing FNN (Sigmoid vs LeakyReLU)
class AssetPricingNet(nn.Module):
    def __init__(self, activation_type='sigmoid'):
        super(AssetPricingNet, self).__init__()
        
        act = nn.Sigmoid() if activation_type == 'sigmoid' else nn.LeakyReLU(negative_slope=0.05)
        
        self.network = nn.Sequential(
            nn.Linear(4, 32),
            act,
            nn.Linear(32, 16),
            act,
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.network(x)

# 3. Train Both Models to See Optimization Under Volatile Constraints
def train_model(model, X, y, epochs=300):
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    loss_history = []
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        predictions = model(X)
        loss = criterion(predictions, y)
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
        
    return loss_history, model

# Run the pricing simulation
sigmoid_model = AssetPricingNet(activation_type='sigmoid')
leaky_model = AssetPricingNet(activation_type='leaky')

sig_losses, trained_sig_model = train_model(sigmoid_model, X, y)
leaky_losses, trained_leaky_model = train_model(leaky_model, X, y)

# 4. Analyze Model Predictions vs Reality on Extreme Tail Events
trained_sig_model.eval()
trained_leaky_model.eval()
with torch.no_grad():
    pred_sig = trained_sig_model(X).numpy().flatten()
    pred_leaky = trained_leaky_model(X).numpy().flatten()
    actual = y.numpy().flatten()

# Filter for the most volatile tail-risk market days (Top 5% extreme targets)
tail_indices = np.where(np.abs(actual) > np.percentile(np.abs(actual), 95))[0][:40]

# 5. Plotting the Financial Dashboards
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Convergence Speed (Loss Curve)
ax1.plot(sig_losses, color='crimson', linewidth=2, label='Sigmoid Net (Saturates on Outliers)')
ax1.plot(leaky_losses, color='darkgreen', linewidth=2, label='LeakyReLU Net (Handles Market Tails)')
ax1.set_title('Training Convergence on Volatile Asset Data', fontsize=12, fontweight='bold')
ax1.set_xlabel('Training Epochs')
ax1.set_ylabel('Mean Squared Error (MSE)')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Subplot 2: Outlier Tail-Event Pricing Capabilities
ax2.scatter(range(len(tail_indices)), actual[tail_indices], color='black', marker='o', s=50, label='True Option Premium')
ax2.plot(pred_sig[tail_indices], color='crimson', linestyle='--', marker='x', label='Sigmoid Pricing Forecast')
ax2.plot(pred_leaky[tail_indices], color='darkgreen', linestyle='-', marker='^', label='LeakyReLU Pricing Forecast')
ax2.set_title('Pricing Performance During Tail-Risk Shock Days', fontsize=12, fontweight='bold')
ax2.set_xlabel('Selected Extreme Trading Days')
ax2.set_ylabel('Asset Premium Valuation')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()


## Detecting AI generated text (Paper by Ren, Raghavan and Garg (2026))

In [ ]:
"""
Neural network detector for AI-generated text under distribution shift, and test-time adaptation 
with unlabeled data.

Idea of the paper
  Ren, Raghavan and Garg (2026), Hitting a Moving Target, Test-Time Adaptation for AI Text Detection
  under Continual Distribution Shift. A supervised detector is trained on human text and the output of
  one LLM. When a new LLM appears it fails. Test-time adaptation (TTA) uses unlabeled text from the
  new LLM, together with known human text, and recovers.

Data
  The paper uses arXiv abstracts (Kaggle, Cornell University) and AI rewrites of them by 11 LLMs, plus
  the RAID benchmark. Those rewrites are not published with the code, so this demo uses the M4 data
  (Wang et al., EACL 2024, github.com/mbzuai-nlp/M4). It contains the same kind of data, 3000 real
  arXiv abstracts and for each of them a rewrite by six generators, ChatGPT, davinci-003, Cohere,
  Dolly, Flan-T5 and BLOOMZ. The files are downloaded on the first run, about 80 MB.

Network
  The multilayer perceptron of the session, written out in numpy. Examples are the columns of X,
  ReLU hidden layer with He initialisation, sigmoid output, cross entropy, so dZ[L] = A - Y,
  backpropagation as in the lecture, and the Adam optimiser.

Test-time adaptation, simplified from the (TED)^n method of the paper
  1. Train the network to separate labeled human text from an unlabeled mixture of human and new-LLM text.
  2. Flag the highest scoring fraction alpha of the unlabeled texts as AI. Alpha is taken as known here.
  3. Retrain on labeled human text and these pseudo-labels.

Run
  python ai_text_shift_demo.py          (or paste it into a notebook cell)
Needs numpy, scikit-learn, matplotlib.
"""
import json, os, time, urllib.request
import numpy as np
import matplotlib

try:                                    # inside Jupyter the inline backend is used
    get_ipython().run_line_magic("matplotlib", "inline")
    IN_IPYTHON = True
except Exception:
    IN_IPYTHON = False
    matplotlib.use("Agg")               # plain script, write the file without a window
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------------ settings
DATA_DIR = "m4_data"
URL = "https://raw.githubusercontent.com/mbzuai-nlp/M4/main/data/arxiv_{}.jsonl"
GENERATORS = {"ChatGPT": "chatGPT", "davinci-003": "davinci", "Cohere": "cohere",
              "Dolly": "dolly", "Flan-T5": "flant5", "BLOOMZ": "bloomz"}
TRAIN_GENERATOR = "ChatGPT"             # the only AI source the supervised detector sees
SEEDS = 3                               # repetitions with different splits, use 1 for a quick look
N_FEATURES, HIDDEN, EPOCHS, LR, WD = 4000, 64, 200, 1e-3, 1e-4
ALPHA = 0.5                             # share of AI text in the unlabeled pool

DK, AC, TL, GR = "#262626", "#B5481F", "#2A7F8E", "#8C8C8C"


# ------------------------------------------------------------------ data
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_m4():
    """Returns human[id] and machine[generator][id], the same 3000 abstracts for every generator."""
    os.makedirs(DATA_DIR, exist_ok=True)
    human, machine = {}, {}
    for name, key in GENERATORS.items():
        path = os.path.join(DATA_DIR, f"arxiv_{key}.jsonl")
        if not os.path.exists(path):
            print(f"downloading {key} ...")
            urllib.request.urlretrieve(URL.format(key), path)
        rows = read_jsonl(path)
        machine[name] = {}
        for r in rows:
            i = r.get("source_ID") or r.get("source_id")
            if key == "bloomz":         # different column names in this file
                human.setdefault(i, r["abstract"].strip())
                machine[name][i] = r["machine_abstract"].strip()
            else:
                human.setdefault(i, r["human_text"].strip())
                machine[name][i] = r["machine_text"].strip()
    return human, machine


# ------------------------------------------------------------------ the network of the session
def he(m_out, m_in, rng):
    return (rng.standard_normal((m_out, m_in)) * np.sqrt(2 / m_in)).astype(np.float32)


def forward(params, X):
    W1, b1, W2, b2 = params
    Z1 = W1 @ X + b1
    A1 = np.maximum(Z1, 0)                                        # ReLU
    Z2 = W2 @ A1 + b2
    A2 = 1 / (1 + np.exp(-np.clip(Z2, -30, 30)))                  # sigmoid
    return Z1, A1, A2


def train(X, y, rng, weights=None):
    """Full-batch Adam on the cross entropy. X has one example per column, shape (features, n)."""
    n = X.shape[1]
    params = [he(HIDDEN, X.shape[0], rng), np.zeros((HIDDEN, 1), np.float32),
              0.5 * he(1, HIDDEN, rng), np.zeros((1, 1), np.float32)]
    m = [np.zeros_like(p) for p in params]
    v = [np.zeros_like(p) for p in params]
    w = np.ones(n, np.float32) if weights is None else (weights / weights.mean()).astype(np.float32)
    for t in range(1, EPOCHS + 1):
        Z1, A1, A2 = forward(params, X)
        dZ2 = (A2 - y[None, :]) * w[None, :]                      # dZ[L] = A - Y
        dA1 = params[2].T @ dZ2                                   # W[2]^T dZ[2]
        dZ1 = dA1 * (Z1 > 0)                                      # times s'(Z[1])
        grads = [dZ1 @ X.T / n + WD * params[0], dZ1.sum(1, keepdims=True) / n,
                 dZ2 @ A1.T / n + WD * params[2], dZ2.sum(1, keepdims=True) / n]
        for k, (p, g) in enumerate(zip(params, grads)):           # Adam
            m[k] = 0.9 * m[k] + 0.1 * g
            v[k] = 0.999 * v[k] + 0.001 * g ** 2
            p -= LR * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
    return params


def score(params, X):
    return forward(params, X)[2][0]


# ------------------------------------------------------------------ one repetition
def metrics(p_human, p_ai, thr=0.5):
    auc = roc_auc_score(np.r_[np.zeros(len(p_human)), np.ones(len(p_ai))], np.r_[p_human, p_ai])
    return dict(ai=np.mean(p_ai > thr), human=np.mean(p_human <= thr), auc=auc)


def one_run(human, machine, seed):
    rng = np.random.default_rng(seed)
    ids = np.array(sorted(human))
    rng.shuffle(ids)
    S, Lab, Uh, Ua, E = ids[:1000], ids[1000:1500], ids[1500:2000], ids[2000:2500], ids[2500:]
    pick = lambda src, idx: [src[i] for i in idx]
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=N_FEATURES, sublinear_tf=True, min_df=3)
    vec.fit(pick(human, S) + pick(machine[TRAIN_GENERATOR], S))   # vocabulary from the source data only
    T = lambda texts: vec.transform(texts).toarray().T.astype(np.float32)

    # supervised detector, human against the training generator
    Xs = np.hstack([T(pick(human, S)), T(pick(machine[TRAIN_GENERATOR], S))])
    ys = np.r_[np.zeros(len(S)), np.ones(len(S))].astype(np.float32)
    sup = train(Xs, ys, rng)

    XL, Xh_e = T(pick(human, Lab)), T(pick(human, E))
    out = {}
    for g in machine:
        Xa_e = T(pick(machine[g], E))
        res = {"supervised": metrics(score(sup, Xh_e), score(sup, Xa_e))}
        if g != TRAIN_GENERATOR:
            # unlabeled pool of the new generator, half human, half AI, different abstracts
            XU = np.hstack([T(pick(human, Uh)), T(pick(machine[g], Ua))])
            X1 = np.hstack([XL, XU])
            y1 = np.r_[np.zeros(XL.shape[1]), np.ones(XU.shape[1])].astype(np.float32)
            w1 = np.r_[np.full(XL.shape[1], XU.shape[1] / XL.shape[1]), np.ones(XU.shape[1])]
            net1 = train(X1, y1, rng, weights=w1)                 # step 1, labeled human against unlabeled
            su = score(net1, XU)
            pseudo = (su > np.quantile(su, 1 - ALPHA)).astype(np.float32)   # step 2
            net2 = train(X1, np.r_[np.zeros(XL.shape[1]), pseudo].astype(np.float32), rng)   # step 3
            res["TTA"] = metrics(score(net2, Xh_e), score(net2, Xa_e))
        out[g] = res
    return out


# ------------------------------------------------------------------ main
def main():
    t0 = time.time()
    human, machine = load_m4()
    i0 = sorted(human)[3]
    print("\nOne abstract, human and machine versions")
    print("human     :", human[i0][:230].replace("\n", " "), "...")
    for g in ("ChatGPT", "Dolly"):
        print(f"{g:10s}:", machine[g][i0][:230].replace("\n", " "), "...")

    runs = [one_run(human, machine, s) for s in range(SEEDS)]
    print(f"\nAI recall, human recall and AUC on held-out abstracts, mean over {SEEDS} splits, threshold 0.5")
    print(f"{'generator':12s}{'method':12s}{'AI recall':>10s}{'human recall':>14s}{'AUC':>8s}")
    summary = {}
    for g in GENERATORS:
        for method in ("supervised", "TTA"):
            if method not in runs[0][g]:
                continue
            vals = {k: np.array([r[g][method][k] for r in runs]) for k in ("ai", "human", "auc")}
            summary[(g, method)] = vals
            print(f"{g:12s}{method:12s}{vals['ai'].mean():10.2f}{vals['human'].mean():14.2f}{vals['auc'].mean():8.3f}")

    # figure
    gens = list(GENERATORS)
    x = np.arange(len(gens)); w = 0.38
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8))
    for a, key, title in ((ax[0], "ai", "Share of AI text detected"), (ax[1], "auc", "AUC")):
        s = [summary[(g, "supervised")][key] for g in gens]
        a.bar(x - w / 2, [v.mean() for v in s], w, yerr=[v.std() for v in s], color=GR, label="supervised, trained on ChatGPT")
        tta = [summary.get((g, "TTA"), {}).get(key) for g in gens]
        a.bar(x[1:] + w / 2, [v.mean() for v in tta[1:]], w, yerr=[v.std() for v in tta[1:]], color=AC, label="test-time adaptation")
        a.set_xticks(x); a.set_xticklabels(gens, fontsize=8.5); a.set_ylim(0, 1.08)
        a.set_title(title, fontsize=11); a.spines[["top", "right"]].set_visible(False); a.grid(axis="y", alpha=0.25)
    ax[0].axvspan(-0.5, 0.5, color=TL, alpha=0.08)
    ax[0].text(0, 1.02, "training\ngenerator", ha="center", va="bottom", fontsize=7.5, color=TL)
    h, l = ax[0].get_legend_handles_labels()
    fig.legend(h, l, fontsize=8.5, frameon=False, loc="lower center", ncol=2)
    fig.suptitle("Detector of AI-written arXiv abstracts, new generators appear after training", fontsize=11, y=1.0)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.savefig("ai_text_shift_demo.png", dpi=200)
    print(f"\nfigure saved to ai_text_shift_demo.png, {time.time() - t0:.0f} s")
    plt.show()


if __name__ == "__main__":
    main()